# 07. 贡献者实战路线：如何读、改、测一个 Feature

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 一条推荐工作流

1. 从 docs 找到用户 API：参数名、endpoint、示例。
2. 在 `ServerArgs` / `io_struct` / OpenAI protocol 中找输入字段。
3. 跟踪字段进入 `TokenizerManager`、`Req`、`ScheduleBatch`。
4. 找 scheduler 决策点：入队、batch formation、run_batch、process result。
5. 找执行点：model runner、attention backend、sampling、grammar/spec/cache worker。
6. 找测试：unit test、kit、server fixture、nightly/performance test。
7. 写最小复现，再写覆盖真实边界条件的测试。


### `Req` 字段是 Feature 进入 scheduler 的常见落点

```python
# python/sglang/srt/managers/schedule_batch.py:666-745
 666: class Req(ReqDllmMixin):
 667:     """The input and output status of a request."""
 668: 
 669:     def __init__(
 670:         self,
 671:         rid: str,
 672:         origin_input_text: str,
 673:         origin_input_ids: array[int],
 674:         sampling_params: SamplingParams,
 675:         return_logprob: bool = False,
 676:         top_logprobs_num: int = 0,
 677:         dllm_config: Optional[DllmConfig] = None,
 678:         token_ids_logprob: List[int] = None,
 679:         stream: bool = False,
 680:         origin_input_ids_unpadded: Optional[array[int]] = None,
 681:         lora_id: Optional[str] = None,
 682:         input_embeds: Optional[List[List[float]]] = None,
 683:         positional_embed_overrides: Optional[PositionalEmbeds] = None,
 684:         token_type_ids: List[int] = None,
 685:         session: Optional[Session] = None,
 686:         custom_logit_processor: Optional[str] = None,
 687:         require_reasoning: bool = False,
 688:         return_hidden_states: bool = False,
 689:         return_routed_experts: bool = False,
 690:         routed_experts_start_len: int = 0,
 691:         return_indexer_topk: bool = False,
 692:         eos_token_ids: Optional[Set[int]] = None,
 693:         bootstrap_host: Optional[str] = None,
 694:         bootstrap_port: Optional[int] = None,
 695:         bootstrap_room: Optional[int] = None,
 696:         disagg_mode: Optional[DisaggregationMode] = None,
 697:         routed_dp_rank: Optional[int] = None,
 698:         disagg_prefill_dp_rank: Optional[int] = None,
 699:         vocab_size: Optional[int] = None,
 700:         priority: Optional[int] = None,
 701:         metrics_collector: Optional[SchedulerMetricsCollector] = None,
 702:         extra_key: Optional[str] = None,
 703:         routing_key: Optional[str] = None,
 704:         dimensions: Optional[int] = None,
 705:         http_worker_ipc: Optional[str] = None,
 706:         time_stats: Optional[
 707:             Union[APIServerReqTimeStats, DPControllerReqTimeStats]
 708:         ] = None,
 709:         return_pooled_hidden_states: bool = False,
 710:         multi_item_delimiter_indices: Optional[List[int]] = None,
 711:     ):
 712:         # Input and output info
 713:         self.rid = rid
      # 注：请求全局 id，是 scheduler 输出回到 TokenizerManager `rid_to_state` 的路由键。
 714:         self.origin_input_ids = origin_input_ids
      # 注：原始 prompt token ids；prefix cache、长度校验和 prefill 都以它为基础。
 715:         self.origin_input_ids_unpadded = (
      # 注：多模态 padding 前的原始 token ids，用于需要还原用户输入长度的统计或返回。
 716:             origin_input_ids_unpadded
 717:             if origin_input_ids_unpadded
 718:             else self.origin_input_ids
 719:         )  # Before image padding
 720:         # Each decode stage's output ids. Append-only by contract:
 721:         # _refresh_fill_ids infers how many output tokens are already in
 722:         # full_untruncated_fill_ids from lengths alone, so in-place rewrites
 723:         # that preserve length would silently corrupt fill_ids.
 724:         self.output_ids = array("q")
      # 注：生成阶段 append-only 的输出 token；scheduler 根据长度差推断新增 token，不能原地改写。
 725:         # Full untruncated sequence: origin + output (+ DLLM mask block).
 726:         # Kept in sync by _refresh_fill_ids; admission only updates fill_len,
 727:         # never mutates this array's length.
 728:         self.full_untruncated_fill_ids = array("q")
      # 注：prompt+output 的完整序列镜像，chunked prefill/DLLM 等路径用它恢复 fill 状态。
 729:         self.fill_len: int = 0
      # 注：当前已经进入 fill/prefill 处理的 token 长度，admission 不直接截断完整序列。
 730: 
 731:         self.session = session
 732:         self.input_embeds = input_embeds
 733:         self.positional_embed_overrides = positional_embed_overrides
 734:         self.multi_item_delimiter_indices = multi_item_delimiter_indices
 735: 
 736:         # For req-level memory management
 737:         self.kv_committed_len = 0
      # 注：已提交、语义上可保留的 KV 长度；finished/cache 路径只处理这部分 KV。
 738:         self.kv_allocated_len = 0
      # 注：已经向 KV pool 申请的长度，可能大于 committed 长度，用于处理预分配和回收。
 739:         self.kv_committed_freed = False
      # 注：记录 committed KV 是否已释放，防止 abort/finish 多路径重复 free。
 740:         self.kv_overallocated_freed = False
      # 注：记录 overallocated KV 是否已释放，避免预分配槽位泄漏或重复释放。
 741: 
 742:         # for corss-endoder model
 743:         self.token_type_ids = token_type_ids
 744: 
 745:         # The length of KV that have been removed in swa cache.
```

**读这段时抓住：**

- 如果你的 Feature 是每请求语义，先问它是否需要成为 `Req` 字段。
- 字段一旦进 `Req`，就要考虑 batch 合并、streaming 返回、abort 清理、测试 fixture。
- 不要把 API 层字段直接塞进 model runner；中间的 scheduler 状态会决定它是否能安全跨进程/跨 batch。


### `BasePrefixCache` 是 cache 类 Feature 的稳定接口

```python
# python/sglang/srt/mem_cache/base_prefix_cache.py:172-245
 172:                             loaded back to device. Pure-KV cache semantics;
 173:         swa_host_hit_length  :   Number of SWA tokens that hit on host (within the sliding
 174:                             window) and will be load-back into the SWA device pool.
 175:         mamba_host_hit_length:   Number of Mamba slots that hit on host and will be load-back
 176:                             into the Mamba device pool. Typically 0 or 1.
 177:         mamba_branching_seqlen: The mamba radix cache branching point, which is the longest
 178:                                 page-aligned position that could've been cache hit if there
 179:                                 exists a mamba state.
 180:     """
 181: 
 182:     device_indices: torch.Tensor
 183:     last_device_node: Any
 184:     last_host_node: Any
 185:     best_match_node: Any
 186:     host_hit_length: int = 0
 187:     swa_host_hit_length: int = 0
 188:     mamba_host_hit_length: int = 0
 189:     mamba_branching_seqlen: Optional[int] = None
 190:     cache_protected_len: Optional[int] = None
 191: 
 192: 
 193: def zero_match_result(tree_cache, match_result: MatchResult) -> MatchResult:
      # 注：关键调用：`zero_match_result` - 把一次命中强制归零，常用于调试或禁用 radix 命中的实验。
 194:     if tree_cache.is_chunk_cache():
 195:         # Chunk caches' match_prefix already returns a miss; no root_node to walk back to.
 196:         return match_result
 197:     root = tree_cache.root_node
 198:     return match_result._replace(
 199:         # [:0] keeps dtype and device of the original tensor (e.g. CUDA int64)
 200:         # without allocating a fresh empty tensor.
 201:         device_indices=match_result.device_indices[:0],
 202:         last_device_node=root,
 203:         last_host_node=root,
 204:         best_match_node=root,
 205:         host_hit_length=0,
 206:         swa_host_hit_length=0,
 207:         mamba_host_hit_length=0,
 208:     )
 209: 
 210: 
 211: class BasePrefixCache(ABC, PrefixCacheTrait):
 212:     """Cache can be indexed by either rid or key."""
 213: 
 214:     metrics_collector: Optional[RadixCacheMetricsCollector] = (
 215:         None  # metrics collector for the cache
 216:     )
 217: 
 218:     def init_metrics_collector(self):
 219:         from sglang.srt.server_args import get_global_server_args
 220: 
 221:         server_args = get_global_server_args()
 222:         labels = {"cache_type": self.__class__.__name__}
 223:         if server_args.extra_metric_labels:
 224:             labels.update(server_args.extra_metric_labels)
 225:         radix_cache_cls = resolve_collector_class(
 226:             server_args,
 227:             STAT_LOGGER_ROLE_RADIX_CACHE,
 228:             RadixCacheMetricsCollector,
 229:         )
 230:         self.metrics_collector = radix_cache_cls(labels=labels)
 231: 
 232:     def update_eviction_metrics(self, num_evicted: int, start_time: float):
 233:         if self.metrics_collector is not None and num_evicted > 0:
 234:             self.metrics_collector.observe_eviction_duration(
 235:                 time.perf_counter() - start_time
 236:             )
 237:             self.metrics_collector.increment_eviction_num_tokens(num_evicted)
 238: 
 239:     @abstractmethod
 240:     def reset(self):
 241:         pass
 242: 
 243:     @abstractmethod
 244:     def match_prefix(self, params: MatchPrefixParams) -> MatchResult:
 245:         pass
```

**读这段时抓住：**

- 新增 cache 实现必须覆盖 match/insert/cache_finished/cache_unfinished/evict/lock-ref 这些生命周期方法。
- `MatchResult` 字段已经为 HiCache/Mamba/SWA 预留多种 hit 类型，新增 cache 行为应优先复用这些语义。
- 如果你只改 RadixCache 而没看 BasePrefixCache，往往会漏掉 ChunkCache/HiRadixCache/UnifiedCache 的兼容性。


## 常用 grep 模式

SGLang 很大，盲读会很累。建议从这些模式开始：


In [ ]:
patterns = [
    "rg -n \"参数名|类名|函数名\" python/sglang docs test",
    "rg -n \"class .*Backend|register_.*backend|REGISTRY\" python/sglang/srt",
    "rg -n \"sampling_params\\.|server_args\\.|req\\.\" python/sglang/srt/managers",
    "rg -n \"cache_finished_req|match_prefix|alloc_for_extend|evict\" python/sglang/srt",
    "rg -n \"SpeculativeAlgorithm|grammar_backend|attention_backend\" python/sglang/srt",
]
for p in patterns:
    print(p)


## 测试入口地图

这个仓库的测试分层明显：

- `python/sglang/test/kits`：可复用测试 kit，很多功能正确性都在这里。
- `python/sglang/test/server_fixtures`：不同 server 配置的 fixture。
- `test/`：项目根目录下也有集成/端到端测试。
- docs notebook：能验证示例，但不应当是唯一测试。


In [ ]:
for d in [
    "python/sglang/test/kits",
    "python/sglang/test/server_fixtures",
    "python/sglang/test",
    "test",
]:
    path = ROOT / d
    if path.exists():
        files = sorted(path.glob("*.py"))
        print(f"\n{d}: {len(files)} python files")
        for f in files[:20]:
            print(" -", f.name)


## 小测试：给 Feature 找测试

下面输入关键词，cell 会找相关测试文件。你可以改 `keyword`。


In [ ]:
keyword = "radix"
roots = [ROOT / "python" / "sglang" / "test", ROOT / "test"]
for root in roots:
    if not root.exists():
        continue
    print("\n==", root.relative_to(ROOT))
    for p in sorted(root.rglob("*.py")):
        text = p.read_text(errors="ignore")
        if keyword.lower() in text.lower() or keyword.lower() in p.name.lower():
            print(p.relative_to(ROOT))


## 修改不同类型 Feature 的最小检查清单

### 新 server 参数

- `server_args.py` 是否定义、校验、默认值合理。
- docs 是否说明默认行为和适用硬件。
- 参数是否参与 metrics / logs，便于线上定位。

### 新调度策略

- waiting queue 和 running batch 的状态是否一致。
- abort、timeout、streaming、grammar pending、chunked prefill 是否安全。
- KV reserve 估计是否保守。

### 新 cache 行为

- prefix key 是否包含隔离维度。
- lock/ref 是否覆盖所有异步路径。
- evict 是否释放了正确 pool。
- page_size > 1 是否正确。

### 新 grammar backend

- grammar object 是否可 copy。
- mask shape、device、vocab size 是否一致。
- jump-forward 是否保持 tokenizer/grammar 状态一致。

### 新 speculative 算法

- accept length、bonus token、KV commit、metrics 是否一致。
- overlap disabled/enabled 是否都能工作，或明确禁止。
- draft/target vocab 不一致时是否报错。


## 建议先读的几个源码片段

如果你只想用一天快速熟悉 SGLang，按这个顺序读：

1. `docs/index.rst`：Feature 总目录。
2. `python/sglang/launch_server.py`：启动分流。
3. `python/sglang/srt/entrypoints/engine.py`：manager 初始化。
4. `python/sglang/srt/managers/tokenizer_manager.py::generate_request`。
5. `python/sglang/srt/managers/scheduler.py::event_loop_*`、`get_new_batch_prefill`、`run_batch`。
6. `python/sglang/srt/managers/schedule_batch.py` 顶部注释和 `Req`。
7. `python/sglang/srt/mem_cache/radix_cache.py`。
8. 任选一个 Feature：structured outputs、speculative、HiCache，按前几本指南深入。


In [ ]:
must_read = [
    "docs/index.rst",
    "python/sglang/launch_server.py",
    "python/sglang/srt/entrypoints/engine.py",
    "python/sglang/srt/managers/tokenizer_manager.py",
    "python/sglang/srt/managers/scheduler.py",
    "python/sglang/srt/managers/schedule_batch.py",
    "python/sglang/srt/mem_cache/radix_cache.py",
]
for p in must_read:
    print(p, "exists=", (ROOT / p).exists())


## 最后的判断标准

你真正理解一个 SGLang Feature，通常意味着你能回答：

- 用户 API 是什么，默认行为是什么？
- 请求对象在哪个字段携带它？
- scheduler 在哪一步看见它？
- 它是否改变 KV cache、batch shape、sampling 或 output streaming？
- 它失败时用户会看到什么错误？
- 它的最小测试和压力测试分别是什么？
